# 08 — ML de clasificación: ¿qué edificios sufren daño severo?
Se entrena con los edificios de **Tainan (Taiwán, 2016)** y **Manta (Ecuador, 2016)**, que tienen daño observado, usando solo variables que también existen en Lima (pisos, periodo, área, Vs30 y la sacudida en el edificio). En Lima no hay daño observado: el champion solo se aplica en `10_riesgo_lima` para ver el efecto en los 6 distritos.
1. **Monitoreo:** si llega una ciudad nueva con daño observado, se mide el AUC del champion en ella y el drift (PSI) de sus variables.
2. **Validación externa (Turquía 2023):** el champion se aplica a los edificios con daño observado en Turquía; se reportan AUC, Brier y drift, sin reentrenar.
3. **Decisión de recalibrado:** si no hay evidencia, el notebook termina y se conserva el champion.
4. **Entrenamiento:** se comparan 4 algoritmos con validación **entre ciudades** (se entrena en una y se prueba en la otra), con pesos que equilibran las clases.
5. **Promoción:** el challenger reemplaza al champion si su AUC entre ciudades mejora al menos 0.01.
6. **MLflow:** la comparación queda registrada como experimento.

## 1. Setup — imports

In [ ]:
# Librerías que usa el notebook.
import json
from datetime import datetime

import mlflow
import numpy as np
import pandas as pd

from pyspark.sql.functions import col, when
from pyspark.sql import functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

## 2. Catálogo, parámetros y umbrales

In [ ]:
# Catálogo y esquema; forzar_recalibrado llega como parámetro del job.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("forzar_recalibrado", "false")
forzar = dbutils.widgets.get("forzar_recalibrado").lower() == "true"

In [ ]:
# Caso, target y umbrales de monitoreo y promoción.
caso = "dano"
target = "etiqueta"
SEED = 700
VARS_EDIFICIO = ["pisos", "Tb", "area_m2"]

UMBRALES = {
    "psi_feature": 0.20,
    "n_features_drift": 2,
    "caida_auc": 0.05,
    "mejora_minima_auc": 0.01,
}

## 3. Función de drift: PSI
Population Stability Index entre la distribución de referencia (datos del champion) y la del lote nuevo. PSI > 0.20 indica un cambio importante.

In [ ]:
# PSI de cada variable entre la referencia y el lote actual.
def psi(referencia, actual, bins=10):
    ref = pd.Series(referencia).dropna().astype(float)
    act = pd.Series(actual).dropna().astype(float)
    cortes = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return 0.0
    cortes[0], cortes[-1] = -np.inf, np.inf
    p = np.histogram(ref, cortes)[0] / len(ref)
    q = np.histogram(act, cortes)[0] / len(act)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

## 4. Datos de Tainan y Manta

In [ ]:
# Lee los datos y las variables seleccionadas.
df = spark.table("fe_dano")

seleccion = spark.table("fe_columnas_seleccionadas").filter(col("caso") == caso)
cols_nums = [r["columna"] for r in seleccion.filter(col("tipo") == "num").collect()]

lotes = sorted(r[0] for r in df.select("ciudad").distinct().collect())
lote_actual = "+".join(lotes)
df.groupBy("ciudad").agg(F.count("*").alias("edificios"), F.round(F.avg(target), 3).alias("fraccion_severo")).show()
print("Variables:", cols_nums)

## 5. Champion actual (registro de modelos)

In [ ]:
# Carga el champion vigente y sus metadatos.
registro = spark.table("ml_registro_modelos").filter(col("caso") == caso)
fila_champion = registro.filter(col("promovido")).orderBy(col("version").desc()).first()

if fila_champion is None:
    champion, meta = None, None
else:
    champion = PipelineModel.load(fila_champion["ruta"])
    meta = {"version": fila_champion["version"],
            "cols_num": json.loads(fila_champion["columnas_num"]),
            "cols_cat": json.loads(fila_champion["columnas_cat"]),
            "snapshots": json.loads(fila_champion["snapshots"]),
            "metricas": json.loads(fila_champion["metricas"]),
            "detalle": json.loads(fila_champion["detalle"])}

print("Champion actual:", "v" + str(meta["version"]) if meta else "no existe")

## 6. Monitoreo: degradación de la métrica y drift

In [ ]:
# Mide la degradación del champion y el drift de cada variable.
n_drift, metrica_base, metrica_nueva, degradacion = 0, None, None, False
evaluador_auc = BinaryClassificationEvaluator(labelCol=target, rawPredictionCol="probability", metricName="areaUnderROC")
nuevas = [c for c in lotes if champion is not None and c not in meta["snapshots"]]

if nuevas:
    ref = df.filter(col("ciudad").isin(meta["snapshots"])).select(meta["cols_num"]).toPandas()
    act = df.filter(col("ciudad").isin(nuevas)).select(meta["cols_num"]).toPandas()
    psis = {c: psi(ref[c], act[c]) for c in meta["cols_num"]}
    n_drift = sum(v > UMBRALES["psi_feature"] for v in psis.values())
    print("PSI (top 5):", [(c, round(v, 3)) for c, v in sorted(psis.items(), key=lambda kv: -kv[1])[:5]])

    metrica_base = meta["metricas"]["AUC"]
    metrica_nueva = evaluador_auc.evaluate(champion.transform(df.filter(col("ciudad").isin(nuevas))))
    degradacion = metrica_base - metrica_nueva > UMBRALES["caida_auc"]
    print(f"AUC base: {metrica_base:.3f} | AUC en {nuevas}: {metrica_nueva:.3f} | degradacion: {degradacion}")
elif champion is not None:
    print("No llegaron ciudades nuevas con daño observado: no hay datos nuevos que monitorear")

## 7. Validación externa: Turquía 2023
El champion se aplica a los edificios de Antakya, Nurdağı y Jindires, que tienen daño visible en imágenes satelitales (interpretación visual; Liu, 2026) y sacudida del ShakeMap del USGS (`fe_dano_turquia`, de `07_demanda_sismica`). Turquía no entra al entrenamiento ni a la decisión de recalibrado, por tres razones: la etiqueta viene de otra fuente, la sacudida también, y sus 24 000 edificios dominarían a los 184 de Tainan y Manta.

Se reportan el AUC, el Brier, la tasa observada frente a la probabilidad media, el porcentaje de edificios fuera del rango de entrenamiento y el PSI frente a Tainan y Manta. El modelo se entrenó con clases equilibradas, así que su probabilidad no está calibrada a la tasa real: la métrica principal es el AUC. Los resultados quedan en `ml_metricas` (notebook `08_ml_dano`) y en MLflow.

In [ ]:
CIUDADES_DOMINIO_TR = ["Antakya", "Nurdagi"]


# Aplica un modelo de daño en Turquía; guarda AUC, Brier, tasas, % fuera de rango y PSI en ml_metricas y MLflow.
def validar_turquia(modelo, meta_modelo):
    if not spark.catalog.tableExists("fe_dano_turquia"):
        print("Sin fe_dano_turquia: no se valida en Turquía (ver 01_bronze)")
        return None
    columnas, rangos = meta_modelo["cols_num"], meta_modelo["detalle"]["rangos"]
    fuera = F.greatest(*[(~col(c).between(*rangos[c])).cast("double") for c in columnas if c in rangos])
    turquia = spark.table("fe_dano_turquia")
    pred = modelo.transform(turquia).withColumn("p", vector_to_array("probability")[1])
    ref = df.select(columnas).toPandas()
    filas = []
    for ciudad in sorted(r[0] for r in turquia.select("ciudad").distinct().collect()):
        d = pred.where(col("ciudad") == ciudad)
        r = d.agg(F.count("*").alias("n"), F.sum(target).alias("pos"), F.avg(target).alias("tasa"), F.avg("p").alias("p_media"),
                  F.avg((col("p") - col(target)) ** 2).alias("brier"), F.avg(fuera).alias("fuera")).first()
        if not r.n:
            continue
        act = turquia.where(col("ciudad") == ciudad).select(columnas).toPandas()
        psis = {c: psi(ref[c], act[c]) for c in columnas}
        filas.append({"ciudad": ciudad, "edificios": r.n, "AUC": evaluador_auc.evaluate(d) if 0 < r.pos < r.n else float("nan"),
                      "brier": r.brier, "tasa_observada": r.tasa, "prob_media_predicha": r.p_media, "pct_fuera_rango": 100 * r.fuera,
                      "n_features_drift": sum(v > UMBRALES["psi_feature"] for v in psis.values()), "psi_max": max(psis.values()),
                      "variable_psi_max": max(psis, key=psis.get)})
    tabla_tr = pd.DataFrame(filas)
    dominio = pred.where(col("ciudad").isin(CIUDADES_DOMINIO_TR))
    auc_dominio = evaluador_auc.evaluate(dominio) if dominio.select(target).distinct().count() == 2 else float("nan")
    print(f"Champion v{meta_modelo['version']}: AUC en {' + '.join(CIUDADES_DOMINIO_TR)} = {auc_dominio:.3f} "
          f"(AUC entre Tainan y Manta = {meta_modelo['metricas']['AUC']:.3f})")
    display(tabla_tr.round(3))

    nombre = f"{caso}_v{meta_modelo['version']}_validacion_turquia"
    params = json.dumps({"fuente_dano": "interpretacion visual (zenodo_18437501)", "sacudida": "ShakeMap USGS us6000jllz", "entrena": False})
    metricas_tr = [(nombre, f"{m}_{f['ciudad']}", float(f[m]), params) for f in filas
                   for m in ("AUC", "brier", "tasa_observada", "prob_media_predicha", "pct_fuera_rango", "n_features_drift")]
    metricas_tr.append((nombre, "AUC_" + "+".join(CIUDADES_DOMINIO_TR), float(auc_dominio), params))
    (spark.createDataFrame(metricas_tr, "modelo string, metrica string, valor double, parametros string")
     .withColumn("fecha", F.lit(datetime.now().isoformat(timespec="seconds"))).withColumn("notebook", F.lit("08_ml_dano"))
     .write.mode("overwrite").option("replaceWhere", "notebook = '08_ml_dano'").saveAsTable("ml_metricas"))

    mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').first()[0]}/sismos_{caso}")
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({"version_modelo": meta_modelo["version"], "ciudades": ",".join(tabla_tr.ciudad), "uso": "validacion externa (no entrena)"})
        metricas_mlflow = {"auc_dominio": float(auc_dominio), **{f"{m.lower()}_{f['ciudad'].lower()}": float(f[m]) for f in filas for m in ("AUC", "brier")}}
        mlflow.log_metrics({k: v for k, v in metricas_mlflow.items() if np.isfinite(v)})
        mlflow.log_text(tabla_tr.to_csv(index=False), "validacion_turquia.csv")
    return tabla_tr

In [ ]:
# Valida el champion vigente en Turquía; no cambia la decisión de recalibrado.
if champion is not None:
    validacion_turquia = validar_turquia(champion, meta)

## 8. Decisión de recalibrado
Se recalibra si no hay champion, si la métrica se degradó, si hay drift en 2 o más variables o si el job lo fuerza. La decisión queda en `ml_monitoreo`.

In [ ]:
# Decide si recalibrar: sin champion, degradación, drift o pedido del job.
if champion is None:
    reentrenar, motivo = True, "No existe champion: primer entrenamiento"
else:
    drift = n_drift >= UMBRALES["n_features_drift"]
    reentrenar = bool(degradacion or drift or forzar)
    senales = [("degradacion de la metrica", degradacion), (f"drift en {n_drift} variables", drift), ("forzado por parametro", forzar)]
    motivos = [texto for texto, activa in senales if activa]
    motivo = " + ".join(motivos) if motivos else "Sin cambios relevantes: se conserva el champion"

print("DECISION:", "RECALIBRAR" if reentrenar else "SIN CAMBIOS", "->", motivo)

In [ ]:
# Guarda la decisión en ml_monitoreo.
clave = f"{lote_actual}|{caso}"
fila = [(clave, lote_actual, caso, meta["version"] if meta else None, int(n_drift), metrica_base, metrica_nueva,
         bool(degradacion), bool(reentrenar), motivo)]
esquema_monitoreo = ("clave string, lote string, caso string, champion_version int, n_features_drift int, "
                     "metrica_base double, metrica_nueva double, degradacion boolean, reentrenar boolean, motivo string")
monitoreo = spark.createDataFrame(fila, esquema_monitoreo)

monitoreo.write.mode("overwrite").option("replaceWhere", f"caso = '{caso}' AND lote = '{lote_actual}'").saveAsTable("ml_monitoreo")

In [ ]:
# Termina aquí si no hace falta recalibrar.
if not reentrenar:
    dbutils.notebook.exit("Sin cambios: se conserva el champion")

## 9. Entrenamiento

In [ ]:
# Agrega pesos para que cada ciudad y clase cuenten por igual.
def con_pesos(datos):
    frac = datos.agg(F.avg(target)).first()[0]
    return datos.withColumn("peso", when(col(target) == 1, 0.5 / frac).otherwise(0.5 / (1 - frac)))


# Pipeline de ML: vector, escalado y algoritmo.
def pipeline(estimador, columnas):
    return Pipeline(stages=[VectorAssembler(inputCols=columnas, outputCol="num_features", handleInvalid="skip"),
                            StandardScaler(inputCol="num_features", outputCol="features_scaled", withMean=True, withStd=True),
                            estimador])


# AUC entrenando en una ciudad y evaluando en la otra.
def auc_entre_ciudades(estimador, columnas):
    aucs = {}
    for prueba in lotes:
        entreno = df.filter(col("ciudad") != prueba)
        modelo = pipeline(estimador, columnas).fit(con_pesos(entreno))
        aucs[prueba] = evaluador_auc.evaluate(modelo.transform(df.filter(col("ciudad") == prueba)))
    return aucs


snapshots_train = lotes
print("Validación: se entrena sin una ciudad y se prueba en ella ->", lotes)

In [ ]:
# Algoritmos candidatos, con los mismos datos y pesos.
comunes = dict(labelCol=target, featuresCol="features_scaled", weightCol="peso")

algoritmos = {
    "LogisticRegression": LogisticRegression(maxIter=100, **comunes),
    "DecisionTree": DecisionTreeClassifier(maxDepth=5, minInstancesPerNode=5, seed=SEED, **comunes),
    "RandomForest": RandomForestClassifier(numTrees=300, minInstancesPerNode=5, seed=SEED, **comunes),
    "GBT": GBTClassifier(maxIter=30, maxDepth=3, seed=SEED, **comunes),
}

## 10. Evaluación entre ciudades

In [ ]:
# Compara algoritmos entrenando en una ciudad y evaluando en la otra.
filas = []
for nombre, estimador in algoritmos.items():
    aucs = auc_entre_ciudades(estimador, cols_nums)
    filas.append({"Modelo": nombre, "AUC": float(np.mean(list(aucs.values()))), **{f"AUC_prueba_{c}": v for c, v in aucs.items()}})
    print("evaluado:", nombre, {c: round(v, 3) for c, v in aucs.items()})

tabla = pd.DataFrame(filas).sort_values("AUC", ascending=False).reset_index(drop=True)
display(tabla.round(4))

## 11. Promoción champion / challenger

In [ ]:
# Elige el ganador y lo contrasta con un modelo solo de variables del edificio.
ganador = tabla.loc[0, "Modelo"]
auc_ganador = float(tabla.loc[0, "AUC"])
auc_solo_edificio = auc_entre_ciudades(algoritmos[ganador], VARS_EDIFICIO)
print(f"{ganador}: AUC entre ciudades = {auc_ganador:.3f} | solo variables del edificio = {np.mean(list(auc_solo_edificio.values())):.3f}")

if champion is None:
    promover = True
else:
    auc_champion = float(meta["metricas"]["AUC"])
    promover = auc_ganador >= auc_champion + UMBRALES["mejora_minima_auc"]

print("Promover challenger a champion:", promover)

## 12. Guardar el modelo en el Volume y registrarlo

In [ ]:
# Entrena con las dos ciudades y guarda rangos e importancias.
modelo_final = pipeline(algoritmos[ganador], cols_nums).fit(con_pesos(df))
etapa = modelo_final.stages[-1]
pesos = etapa.featureImportances.toArray() if hasattr(etapa, "featureImportances") else etapa.coefficients.toArray()
importancia = pd.Series(pesos, index=cols_nums).sort_values(key=abs, ascending=False)
display(importancia.round(3).to_frame("importancia_o_coeficiente"))

rangos = {c: [float(v) for v in df.agg(F.min(c), F.max(c)).first()] for c in cols_nums}
metricas = {"AUC": auc_ganador, **{k: float(v) for k, v in tabla.loc[0].items() if k.startswith("AUC_prueba_")}}
detalle = {"comparacion": tabla.round(4).to_dict("records"), "auc_solo_edificio": auc_solo_edificio,
           "importancia": importancia.round(4).to_dict(), "rangos": rangos}

In [ ]:
# Guarda el modelo en el Volume y lo registra.
version = registro.count() + 1
ruta = f"{volumen}/modelos/{caso}/v{version}"
modelo_final.write().overwrite().save(ruta)

fila = [(caso, version, ganador, lote_actual, ruta, bool(promover), json.dumps(cols_nums), json.dumps([]),
         json.dumps(snapshots_train), json.dumps(metricas), json.dumps(detalle, default=float))]
esquema_registro = ("caso string, version int, algoritmo string, lote string, ruta string, promovido boolean, "
                    "columnas_num string, columnas_cat string, snapshots string, metricas string, detalle string")
spark.createDataFrame(fila, esquema_registro).write.mode("append").saveAsTable("ml_registro_modelos")

print(f"Version v{version} guardada en {ruta}")
print("Estado:", "CHAMPION" if promover else "challenger (se conserva el champion actual)")

## 13. MLflow — experimento
Cada ejecución del notebook queda registrada en un experimento de MLflow (un *run* padre por ejecución y un *run* hijo por algoritmo) para comparar parámetros y métricas desde la interfaz.

In [ ]:
# Experimento de MLflow del caso.
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/sismos_{caso}")

In [ ]:
# Registra parámetros, métricas y modelo en MLflow.
with mlflow.start_run(run_name=f"{caso}_{lote_actual}"):
    mlflow.log_params({"ciudades": lote_actual, "forzado": forzar, "n_edificios": df.count(), "variables": ",".join(cols_nums),
                       "ganador": ganador, "promovido": bool(promover), "version_registro": version, "ruta_modelo": ruta})
    mlflow.log_metric("auc_ganador", auc_ganador)
    if champion is not None:
        mlflow.log_metric("auc_champion", auc_champion)
    mlflow.log_text(tabla.to_csv(index=False), "comparacion.csv")

    for _, fila_modelo in tabla.iterrows():
        with mlflow.start_run(run_name=fila_modelo["Modelo"], nested=True):
            mlflow.log_param("algoritmo", fila_modelo["Modelo"])
            mlflow.log_metrics({k.lower(): float(v) for k, v in fila_modelo.items() if k.startswith("AUC")})

## 14. Validación en Turquía del nuevo champion
Si el challenger se promovió, se repite la validación externa con él y sus métricas reemplazan a las del champion anterior en `ml_metricas`.

In [ ]:
# Valida en Turquía el modelo recién promovido.
if promover:
    validacion_turquia = validar_turquia(modelo_final, {"version": version, "cols_num": cols_nums, "metricas": metricas, "detalle": detalle})